# 01 — Data Preprocessing

This notebook walks through the Smartflat data preprocessing pipeline (Chapter 4).

**Pipeline overview:**
1. **Raw Data Discovery** — Scan data roots for video files, extract metadata (fps, size, duration)
2. **Folder Validation** — Standardize directory structure, fix modality naming
3. **Metadata Creation** — Build the base dataset metadata from consolidated folders
4. **Audio Extraction** — Extract .wav audio tracks from merged videos
5. **Cross-Modality Synchronization** — Compute audio cross-correlation lags between GoPro and Tobii
6. **Video Collation** — Merge multi-partition videos into single files via ffmpeg
7. **Gold Filtering** — Apply quality control and gold data criteria for the final dataset

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from smartflat.datasets.filter import apply_gold_data_filtering
from smartflat.datasets.loader import get_dataset
from smartflat.datasets.quality_control import (
    apply_visual_inspection_results,
    create_annotation_summary,
)
from smartflat.features.consolidation.main_audio_extraction import (
    main as main_audio_extraction,
)
from smartflat.features.consolidation.main_collate import main as main_collate
from smartflat.features.consolidation.main_consolidation import (
    consolidate_dataset,
    consolidate_metadata,
)
from smartflat.features.consolidation.main_registration import (
    main as main_registration,
)
from smartflat.features.consolidation.main_synchronisation import (
    main as main_synchronisation,
)
from smartflat.utils.utils_io import get_data_root

In [ ]:
# Data root — resolved via hostname or SMARTFLAT_DATA_ROOT env var
data_root = get_data_root()
print(f"Data root: {data_root}")

## 1. Raw Data Discovery

Scan all configured data roots for video files and register them in a single metadata CSV.
The registration extracts per-video metadata (fps, frame count, file size, creation date),
parses participant/task/modality from folder paths, applies manual corrections, and filters
outliers (size < 0.05 GB, n_frames < 50, fps < 4).

See: `smartflat.features.consolidation.main_registration`

In [ ]:
# Registration: walk data roots, extract video metadata, save to CSV
# Set path_metadata to override the default output location
path_metadata = os.path.join(data_root, 'dataframes', 'persistent_metadata', 'metadata_raw.csv')
print(f"Registration output: {path_metadata}")

# Uncomment to run (scans all root_paths, takes a few minutes):
# main_registration(path_metadata=path_metadata)

In [ ]:
# Inspect the raw registration metadata
if os.path.isfile(path_metadata):
    df_raw = pd.read_csv(path_metadata)
    print(f"Raw metadata: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
    print(f"Unique participants: {df_raw['participant_id'].nunique()}")
    print(f"Unique modalities:   {df_raw['modality'].nunique()}")
    print(f"\nVideos per task:")
    display(df_raw.groupby('task_name').agg(
        n_videos=('video_path', 'count'),
        n_participants=('participant_id', 'nunique'),
    ))
    print(f"\nVideos per modality:")
    display(df_raw['modality'].value_counts().to_frame('n_videos'))
else:
    print(f"Raw metadata not found at {path_metadata}. Run main_registration() first.")

## 2. Folder Validation

Standardize directory structure by renaming misnamed modality folders and enforcing the
expected layout: `{task}/{participant_id}/{modality}/`. Also consolidate the raw metadata
CSV into a clean version with aggregated partition statistics (n_videos, total size, duration).

See: `smartflat.features.consolidation.main_consolidation.consolidate_dataset`,
     `smartflat.features.consolidation.main_consolidation.consolidate_metadata`

In [ ]:
# Dry-run: generate folder rename/move commands without executing
commands = consolidate_dataset(root_dir=data_root, process=False)

if commands:
    print(f"Generated {len(commands)} commands to fix folder structure.")
    print("First 5 commands:")
    for cmd in commands[:5]:
        print(f"  {cmd}")
else:
    print("No folder fixes needed — structure is already clean.")

In [ ]:
# Consolidate raw metadata: aggregate partitions, compute group statistics
metadata_input = os.path.join(data_root, 'dataframes', 'persistent_metadata', 'metadata_raw.csv')
metadata_output = os.path.join(data_root, 'dataframes', 'persistent_metadata', 'metadata.csv')

if os.path.isfile(metadata_input):
    df_consolidated = consolidate_metadata(input_file=metadata_input, output_file=metadata_output)

    if df_consolidated is not None:
        print(f"Consolidated metadata: {df_consolidated.shape[0]} rows, {df_consolidated.shape[1]} columns")
        print(f"\nColumns: {list(df_consolidated.columns)}")
        print(f"\nTask distribution:")
        display(df_consolidated['task_name'].value_counts().to_frame('count'))
        print(f"\nModality distribution:")
        display(df_consolidated['modality'].value_counts().to_frame('count'))
else:
    print(f"Raw metadata not found at {metadata_input}. Run registration first.")

## 3. Metadata Creation

Build the base dataset by scanning the consolidated data directory. The `SmartflatDataset`
loader discovers video files, parses feature extraction flags, and assembles a metadata
DataFrame with columns for paths, fps, duration, processing status, and more.

See: `smartflat.datasets.loader.get_dataset`, `smartflat.datasets.base_dataset.SmartflatDataset`

In [ ]:
# Load the base dataset (scenario='all' includes all videos regardless of processing status)
dset = get_dataset(dataset_name='base', scenario='all')
df = dset.metadata.copy()

print(f"Base dataset: {len(df)} rows")
print(f"Unique participants: {df['participant_id'].nunique()}")
display(df[['identifier', 'participant_id', 'task_name', 'modality',
            'video_name', 'fps', 'duration']].head(10))

In [ ]:
# Summary statistics: recordings per task/modality, duration distribution
print("Recordings per task × modality:")
display(df.groupby(['task_name', 'modality']).agg(
    n_recordings=('identifier', 'nunique'),
    mean_duration_min=('duration', 'mean'),
    total_size_gb=('size', lambda x: x.sum() if 'size' in df.columns else np.nan),
).round(2))

# FPS distribution
print(f"\nFPS values: {sorted(df['fps'].dropna().unique())}")

# Duration distribution
if df['duration'].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, task in zip(axes, df['task_name'].unique()):
        durations = df[df['task_name'] == task]['duration'].dropna()
        ax.hist(durations, bins=30, edgecolor='black', alpha=0.7)
        ax.set(xlabel='Duration (min)', ylabel='Count',
               title=f'{task} — duration distribution (n={len(durations)})')
        ax.axvline(durations.median(), color='red', ls='--',
                   label=f'Median = {durations.median():.1f} min')
        ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Audio Extraction

Extract .wav audio tracks from consolidated (merged) videos. Audio files are needed for
cross-modality synchronization (step 5). The extraction uses `ffmpeg` via `extract_audio()`
and processes only merged videos or single-partition recordings.

See: `smartflat.features.consolidation.main_audio_extraction`,
     `smartflat.utils.utils_io.extract_audio`

In [ ]:
# Audio extraction: for each merged video, extract a .wav file alongside it
# Uncomment to run (requires ffmpeg, processes all videos):
# main_audio_extraction(root_dir=data_root)

# Check existing audio files
dset_present = get_dataset(dataset_name='base', scenario='present')
df_present = dset_present.metadata.copy()
df_merged = df_present[(df_present['video_name'] == 'merged_video') | (df_present['n_videos'] == 1)]

n_with_audio = 0
for _, row in df_merged.iterrows():
    if pd.notna(row.get('video_path')):
        audio_path = row['video_path'].replace('.mp4', '.wav')
        if os.path.isfile(audio_path):
            n_with_audio += 1

print(f"Merged/single videos: {len(df_merged)}")
print(f"With extracted audio:  {n_with_audio}")
print(f"Missing audio:         {len(df_merged) - n_with_audio}")

## 5. Cross-Modality Synchronization

Compute temporal alignment between GoPro and Tobii recordings using audio cross-correlation.
For each participant, all pairwise modality combinations are aligned by finding the lag that
maximizes the cross-correlation of their audio tracks. Triangle inequality constraints are
checked to validate consistency (e.g., lag(A→C) ≈ lag(A→B) + lag(B→C)).

See: `smartflat.features.consolidation.main_synchronisation`

In [ ]:
# Synchronization: compute cross-correlation lags for all modality pairs
# Uncomment to run (requires audio files from step 4, takes several minutes):
# results_sync = main_synchronisation(root_dir=data_root, verbose=False)

# Load pre-computed synchronization results
sync_dir = os.path.join(data_root, 'dataframes', 'synchronisation')
sync_files = sorted([f for f in os.listdir(sync_dir) if f.startswith('cross_correlation')]) if os.path.isdir(sync_dir) else []

if sync_files:
    sync_path = os.path.join(sync_dir, sync_files[-1])  # most recent
    results_sync = pd.read_csv(sync_path)
    print(f"Synchronization results: {sync_path}")
    print(f"  {results_sync.shape[0]} participants")
    display(results_sync.head(10))
else:
    print("No synchronization results found. Run main_synchronisation() first.")
    results_sync = None

In [ ]:
# Constraint satisfaction and distortion analysis
if results_sync is not None:
    from smartflat.features.consolidation.main_synchronisation import (
        check_constraints,
        compute_max_distorsion,
    )

    results_sync['constraints_ok'] = results_sync.apply(check_constraints, axis=1)
    results_sync['max_distortion'] = results_sync.apply(compute_max_distorsion, axis=1)

    n_ok = results_sync['constraints_ok'].sum()
    n_total = len(results_sync)
    print(f"Triangle inequality constraints satisfied: {n_ok}/{n_total} ({n_ok/n_total:.1%})")
    print(f"Max distortion: mean={results_sync['max_distortion'].mean():.2f}, "
          f"median={results_sync['max_distortion'].median():.2f}, "
          f"max={results_sync['max_distortion'].max():.2f}")

In [ ]:
# Visualize lag distributions and constraint satisfaction
if results_sync is not None:
    lag_cols = [c for c in results_sync.columns if 'lag' in c.lower() or 'offset' in c.lower()]
    if not lag_cols:
        # Fallback: look for modality-pair columns (e.g., GoPro1_GoPro2, GoPro1_Tobii)
        lag_cols = [c for c in results_sync.columns
                    if any(m in c for m in ['GoPro', 'Tobii']) and results_sync[c].dtype in [np.float64, np.int64]]

    if lag_cols:
        fig, axes = plt.subplots(1, min(len(lag_cols), 3), figsize=(5 * min(len(lag_cols), 3), 4))
        if len(lag_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, lag_cols[:3]):
            vals = results_sync[col].dropna()
            ax.hist(vals, bins=30, edgecolor='black', alpha=0.7)
            ax.set(xlabel='Lag (samples)', ylabel='Count', title=col)
            ax.axvline(vals.median(), color='red', ls='--',
                       label=f'Median = {vals.median():.0f}')
            ax.legend()
        plt.tight_layout()
        plt.show()

    # Distortion histogram
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(results_sync['max_distortion'].dropna(), bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(1.0, color='red', ls='--', label='Margin = 1')
    ax.set(xlabel='Max distortion', ylabel='Count',
           title='Cross-modality synchronization — max distortion')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Video Collation

Merge multi-partition video recordings into a single `merged_video.mp4` per modality using
ffmpeg's concat demuxer. Partitions arise when GoPro cameras split long recordings into
multiple files. After collation, partition files can be cleaned up.

See: `smartflat.features.consolidation.main_collate`

In [ ]:
# Dry-run: show ffmpeg concat commands without executing
# Set n_group to limit the number of groups processed (for testing)
commands_collate = main_collate(root_dir=data_root, process=False, n_group=5)

if commands_collate:
    print(f"Generated {len(commands_collate)} collation commands.")
    print("Example (first command):")
    print(f"  {commands_collate[0]}")
else:
    print("No collation needed — all videos are already merged or single-partition.")

In [ ]:
# Post-collation check: verify merged videos exist
dset_post = get_dataset(dataset_name='base', scenario='present')
df_post = dset_post.metadata.copy()

n_merged = (df_post['video_name'] == 'merged_video').sum()
n_single = ((df_post['video_name'] != 'merged_video') & (df_post['n_videos'] == 1)).sum()
n_partitions = ((df_post['video_name'] != 'merged_video') & (df_post['n_videos'] > 1)).sum()

print(f"Present videos: {len(df_post)}")
print(f"  Merged videos:              {n_merged}")
print(f"  Single-partition (no merge): {n_single}")
print(f"  Residual partitions:         {n_partitions}")

if n_partitions > 0:
    print(f"\nResidual partitions (first 5):")
    display(df_post[(df_post['video_name'] != 'merged_video') & (df_post['n_videos'] > 1)]
            [['identifier', 'participant_id', 'modality', 'video_name', 'n_videos']].head(5))

## 7. Gold Filtering

Apply quality control corrections (visual inspection results: modality swaps, failed collections,
flag reassessment) and gold data criteria (remove partitions when merged video exists, filter
duration outliers, remove missing-fps videos) to produce the final analysis-ready dataset.

See: `smartflat.datasets.quality_control.apply_visual_inspection_results`,
     `smartflat.datasets.filter.apply_gold_data_filtering`

In [ ]:
# Step 1: Apply visual inspection corrections (modality swaps, failed collections, flag updates)
dset_all = get_dataset(dataset_name='base', scenario='all')
df_qc = dset_all.metadata.copy()
print(f"Before QC: {len(df_qc)} rows, {df_qc['participant_id'].nunique()} participants")

df_qc = apply_visual_inspection_results(df_qc, verbose=True)
print(f"After QC:  {len(df_qc)} rows, {df_qc['participant_id'].nunique()} participants")

In [ ]:
# Step 2: Apply gold data filtering (remove partitions, duration outliers, missing fps)
# process=False adds is_outlier flags without removing rows; process=True filters
df_gold_flagged = apply_gold_data_filtering(df_qc, process=False, verbose=True)

# Show outlier breakdown
if 'is_outlier' in df_gold_flagged.columns:
    n_outliers = df_gold_flagged['is_outlier'].sum()
    print(f"\nFlagged as outlier: {n_outliers} / {len(df_gold_flagged)}")

# Apply the filter
df_gold = apply_gold_data_filtering(df_qc, process=True, verbose=True)
print(f"\nGold dataset: {len(df_gold)} rows, {df_gold['participant_id'].nunique()} participants")

In [ ]:
# Final gold dataset summary
print("Gold dataset — recordings per task × modality:")
display(df_gold.groupby(['task_name', 'modality']).agg(
    n_recordings=('identifier', 'nunique'),
    n_participants=('participant_id', 'nunique'),
    mean_duration_min=('duration', 'mean'),
    std_duration_min=('duration', 'std'),
).round(2))

# Feature extraction flag summary
flag_cols = [c for c in df_gold.columns if c.startswith('flag_')]
if flag_cols:
    print(f"\nFeature extraction flags:")
    for col in flag_cols:
        counts = df_gold[col].value_counts()
        print(f"  {col}: {dict(counts)}")

# Duration distribution of the gold dataset
if df_gold['duration'].notna().any():
    fig, ax = plt.subplots(figsize=(10, 4))
    for task in sorted(df_gold['task_name'].unique()):
        durations = df_gold[df_gold['task_name'] == task]['duration'].dropna()
        ax.hist(durations, bins=30, alpha=0.6, label=f'{task} (n={len(durations)})',
                edgecolor='black')
    ax.set(xlabel='Duration (min)', ylabel='Count',
           title=f'Gold dataset — duration distribution ({len(df_gold)} recordings)')
    ax.legend()
    plt.tight_layout()
    plt.show()